# Building the Backend for a Full-Stack CrewAI App

## Step-by-Step Plan: From Notebook to FastAPI Backend

In the notebook `037-5th-top-use-case-crewai-2026.ipynb`, we built an AI-powered customer support system using **CrewAI Flows + Crews**. That notebook runs everything inside Jupyter — you type a ticket, and the AI processes it right there.

But what if we want a **real web application** where users can submit tickets from a browser? For that, we need a **backend** — a server that receives requests from the frontend and runs our CrewAI logic.

**In this notebook, we will build that backend step by step.**

## What Is a Backend?

A **backend** is a program that runs on a server and waits for requests. When a user clicks "Submit Ticket" in the browser (the frontend), the frontend sends a request to the backend. The backend processes it and sends back a response.

```
User's Browser (Frontend)          Server (Backend)
┌─────────────────────┐            ┌─────────────────────┐
│                     │  HTTP POST │                     │
│  Submit Ticket Form │ ─────────> │  FastAPI receives   │
│                     │            │  the ticket data    │
│                     │            │                     │
│                     │  JSON      │  CrewAI processes   │
│  Display Response   │ <───────── │  it and responds    │
│                     │            │                     │
└─────────────────────┘            └─────────────────────┘
```

We'll use **FastAPI** as our backend framework — the same one used in the LangChain todo app (`050-todoapp-LC-backend`).

## The Big Picture: What We're Building

Our backend will have **one main API endpoint**:

| Method | Endpoint | What It Does |
|--------|----------|-------------|
| POST | `/support/ticket` | Receives a support ticket, runs CrewAI, returns the AI response |

That's it! One endpoint. The CrewAI Flow handles all the complexity (classifying, routing, resolving).

### Project Structure

Following the same pattern as the LangChain todo app, our backend folder will look like this:

```
074-crewai-backend/
├── .env                  ← API keys (OpenAI)
├── main.py               ← FastAPI app setup + CORS
├── schemas.py            ← Pydantic models (request/response shapes)
├── crew_logic.py         ← All CrewAI code (agents, tools, flow)
└── routers/
    └── support.py        ← API endpoint for /support/ticket
```

**Why this structure?** It keeps things organized:
- `main.py` — Sets up the server (same role as in the todo app).
- `schemas.py` — Defines what data goes in and out (same role as in the todo app).
- `crew_logic.py` — All the CrewAI logic from our notebook, packaged into a reusable function.
- `routers/support.py` — The API endpoint that connects the frontend to CrewAI (same pattern as `routers/todos.py` in the todo app).

Notice: **no database needed!** The todo app used PostgreSQL because it needed to store todos. Our app just processes tickets and returns responses — no storage required.

## Step 0: Install the Required Packages

We need everything from our CrewAI notebook, plus **FastAPI** and **uvicorn** (the server that runs FastAPI).

In [ ]:
# Install packages (run this once)
#!pip install -U crewai python-dotenv fastapi uvicorn[standard]

## Step 1: The `.env` File

Just like in the todo app, we store our API key in a `.env` file. The backend reads this file on startup.

Create a file called `.env` inside the `074-crewai-backend/` folder with this content:

```
OPENAI_API_KEY=your-openai-api-key-here
```

**Important:** Never commit your `.env` file to git. Add it to `.gitignore`.

## Step 2: `schemas.py` — Define the Request and Response Shapes

In the todo app, `schemas.py` defined `ToDoRequest` (what the frontend sends) and `ToDoResponse` (what the backend sends back).

We do the same thing here. When the frontend submits a support ticket, it sends us these fields:
- `customer_name`
- `customer_email`
- `ticket_subject`
- `ticket_message`

And we send back the AI's response with:
- `category` — what kind of ticket it is
- `complexity` — simple or complex
- `urgency` — low, medium, or high
- `path_taken` — which path CrewAI used
- `resolution_subject` — subject line of the response
- `resolution_message` — the full AI response
- `follow_up_needed` — whether a human should follow up

In [ ]:
%%writefile 074-crewai-backend/schemas.py
from pydantic import BaseModel


# What the frontend sends to us
class TicketRequest(BaseModel):
    customer_name: str
    customer_email: str
    ticket_subject: str
    ticket_message: str


# What we send back to the frontend
class TicketResponse(BaseModel):
    customer_name: str
    customer_email: str
    category: str
    complexity: str
    urgency: str
    path_taken: str
    resolution_subject: str
    resolution_message: str
    follow_up_needed: bool

**Comparison with the todo app:**

| Todo App (`schemas.py`) | Our App (`schemas.py`) |
|------------------------|----------------------|
| `ToDoRequest(name, completed)` | `TicketRequest(customer_name, email, subject, message)` |
| `ToDoResponse(name, completed, id)` | `TicketResponse(category, complexity, urgency, resolution_message, ...)` |

Same pattern, different data.

## Step 3: `crew_logic.py` — The CrewAI Brain

This is the heart of our backend. It contains **all the CrewAI code** from the notebook `037-5th-top-use-case-crewai-2026.ipynb`, packaged into a single function called `process_ticket()`.

Think of it this way:
- In the **notebook**, we ran the flow manually: created a flow, set the state, called `kickoff()`.
- In the **backend**, the `process_ticket()` function does all of that automatically when the API endpoint calls it.

Here's what's inside this file:
1. The FAQ and Knowledge Base data (same as the notebook).
2. The tools: `search_faq` and `search_knowledge_base` (same as the notebook).
3. The agents: Classifier, FAQ Specialist, Researcher, Solution Writer (same as the notebook).
4. The Pydantic models for agent output: `TicketClassification`, `SupportResponse` (same as the notebook).
5. The Flow State: `SupportTicketState` (same as the notebook).
6. The Flow: `CustomerSupportFlow` (same as the notebook).
7. **NEW:** A `process_ticket()` function that wraps everything into an easy-to-call function.

#### Update note: current OpenAI models
* The original example is very good for learning the concept, and it worked with the OpenAI model it was recorded with: `gpt-4o-mini`, with `temperature=0`.
* OpenAI is retiring `gpt-4o-mini` (shutdown planned for January 14, 2027). With the current models, we would use `gpt-6-luna`, the small model OpenAI recommends now. `MODEL` defines it once, so you can change it in one line.
* Recent OpenAI models only accept the default temperature (1), so the code uses `temperature=1`. The idea behind `temperature=0` (more predictable answers, useful to classify tickets the same way every time) is still important, and other models still let you choose it.
* The agents of this flow use tools. Through the endpoint CrewAI uses, `gpt-6-luna` only accepts tools when its reasoning is turned off, so the code adds `extra_body={"reasoning_effort": "none"}`. The model then answers directly, like `gpt-4o-mini` did. It works with the version of CrewAI in the backend repository (1.9.3) and with the latest version that `pip install -U crewai` installs.
* Check available models at developers.openai.com/api/docs/deprecations.

In [ ]:
%%writefile 074-crewai-backend/crew_logic.py
import os
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool
from crewai.flow.flow import Flow, start, listen, router, or_

load_dotenv()

# ============================================================
# 1. KNOWLEDGE BASES (same as the notebook)
# ============================================================

FAQ_DB = {
    "password_reset": {
        "question": "How do I reset my password?",
        "answer": (
            "To reset your password:\n"
            "1. Go to app.bigproject.com/login\n"
            "2. Click 'Forgot Password'\n"
            "3. Enter your email address\n"
            "4. Check your inbox for a reset link (arrives within 2 minutes)\n"
            "5. Click the link and set a new password\n\n"
            "If you don't receive the email, check your spam folder."
        ),
    },
    "pricing": {
        "question": "What are your pricing plans?",
        "answer": (
            "BigProject Pro pricing:\n"
            "- Free: Up to 5 users, basic features\n"
            "- Team: $15/user/month — unlimited projects, integrations\n"
            "- Enterprise: $45/user/month — SSO, audit logs, priority support\n\n"
            "All paid plans include a 14-day free trial. "
            "Visit bigproject.com/pricing for details."
        ),
    },
    "cancel_subscription": {
        "question": "How do I cancel my subscription?",
        "answer": (
            "To cancel your subscription:\n"
            "1. Go to Settings > Billing\n"
            "2. Click 'Manage Subscription'\n"
            "3. Select 'Cancel Plan'\n"
            "4. Confirm cancellation\n\n"
            "Your access continues until the end of the current billing period. "
            "No refunds for partial months."
        ),
    },
    "integrations": {
        "question": "What integrations do you support?",
        "answer": (
            "BigProject Pro integrates with 200+ tools including:\n"
            "- Slack, Microsoft Teams, Discord\n"
            "- GitHub, GitLab, Bitbucket\n"
            "- Jira, Linear, Asana\n"
            "- Google Drive, Dropbox, OneDrive\n\n"
            "Visit Settings > Integrations to connect your tools."
        ),
    },
}

KNOWLEDGE_BASE = {
    "billing": {
        "topic": "Billing & Refund Policies",
        "content": (
            "BILLING POLICIES:\n"
            "- Charges are processed on the 1st of each month.\n"
            "- Double charges can occur if a payment fails and retries.\n"
            "- Refund policy: Full refunds within 30 days of charge.\n"
            "- Partial refunds: Pro-rated for mid-cycle cancellations on annual plans.\n"
            "- To process a refund, a support agent must verify the charge in Stripe.\n"
            "- Refunds take 5-10 business days to appear on the customer's statement.\n"
            "- Escalate disputes over $500 to the billing manager."
        ),
    },
    "technical": {
        "topic": "Technical Troubleshooting",
        "content": (
            "COMMON TECHNICAL ISSUES:\n"
            "- API rate limits: 1000 requests/minute (Team), 5000/minute (Enterprise).\n"
            "- File upload limit: 100MB per file, 10GB total storage (Team).\n"
            "- 500 errors on upload: Usually caused by files exceeding size limit.\n"
            "- Sync issues: Clear browser cache, check internet connection.\n"
            "- SSO login failures: Verify SAML configuration in Settings > Security.\n"
            "- Webhook failures: Check endpoint URL and ensure HTTPS.\n"
            "- For persistent 500 errors, collect the request ID from the error page "
            "and escalate to the engineering team."
        ),
    },
    "account": {
        "topic": "Account Management",
        "content": (
            "ACCOUNT POLICIES:\n"
            "- Account deletion: Permanent, cannot be undone. 30-day grace period.\n"
            "- Data export: Available in Settings > Data > Export (CSV or JSON).\n"
            "- Team transfers: Admin can transfer ownership in Settings > Team.\n"
            "- Suspended accounts: Due to payment failure. Reactivate by updating payment.\n"
            "- GDPR data requests: Must be processed within 30 days."
        ),
    },
}


# ============================================================
# 2. TOOLS (same as the notebook)
# ============================================================

@tool("search_faq")
def search_faq(query: str) -> str:
    """Search the FAQ database for answers to common customer questions.

    Use this tool to find pre-written answers for simple questions about
    password resets, pricing, cancellations, and integrations.

    Args:
        query: A keyword or topic to search for (e.g., 'password', 'pricing').
    """
    query_lower = query.lower()
    results = []
    for key, entry in FAQ_DB.items():
        if (
            query_lower in key
            or query_lower in entry["question"].lower()
            or query_lower in entry["answer"].lower()
        ):
            results.append(f"Q: {entry['question']}\nA: {entry['answer']}")
    if results:
        return "\n\n---\n\n".join(results)
    return f"No FAQ entries found for '{query}'. Try different keywords."


@tool("search_knowledge_base")
def search_knowledge_base(topic: str) -> str:
    """Search the detailed knowledge base for policies and procedures.

    Use this tool to find in-depth information about billing policies,
    technical troubleshooting steps, or account management procedures.

    Args:
        topic: The topic to search for (e.g., 'billing', 'technical', 'account').
    """
    topic_lower = topic.lower()
    results = []
    for key, entry in KNOWLEDGE_BASE.items():
        if (
            topic_lower in key
            or topic_lower in entry["topic"].lower()
            or topic_lower in entry["content"].lower()
        ):
            results.append(f"Topic: {entry['topic']}\n{entry['content']}")
    if results:
        return "\n\n---\n\n".join(results)
    return f"No knowledge base entries found for '{topic}'. Try different keywords."


# ============================================================
# 3. STRUCTURED OUTPUT MODELS (same as the notebook)
# ============================================================

class TicketClassification(BaseModel):
    """Structured output from the Ticket Classifier."""
    category: str = Field(
        description=(
            "Ticket category: 'password_reset', 'pricing', "
            "'cancel_subscription', 'integrations', 'billing_issue', "
            "'technical_issue', or 'account_issue'"
        )
    )
    complexity: str = Field(
        description="'simple' (FAQ can answer) or 'complex' (needs investigation)"
    )
    urgency: str = Field(
        description="'low', 'medium', or 'high'"
    )
    reasoning: str = Field(
        description="Brief explanation of why this classification was chosen"
    )


class SupportResponse(BaseModel):
    """Structured response to send to the customer."""
    subject: str = Field(description="Response subject line")
    message: str = Field(description="The full response message to the customer")
    follow_up_needed: bool = Field(
        description="Whether a human agent should follow up"
    )


# ============================================================
# 4. AGENTS (same as the notebook)
# ============================================================

# Original version: LLM(model="openai/gpt-4o-mini", temperature=0).
# OpenAI is retiring gpt-4o-mini (shutdown planned for January 14, 2027),
# so we use gpt-6-luna, the small model OpenAI recommends now.
# Check available models at developers.openai.com/api/docs/deprecations
MODEL = "openai/gpt-6-luna"

# Recent OpenAI models only accept the default temperature (1), so we use 1
# instead of 0. The agents of this flow use tools, and through the endpoint
# CrewAI uses (Chat Completions), gpt-6-luna only accepts tools when its
# reasoning is turned off, so we set reasoning_effort to "none". We send it
# with extra_body, which works with this project's version of CrewAI and
# with the latest one.
llm = LLM(model=MODEL, temperature=1, extra_body={"reasoning_effort": "none"})

classifier_agent = Agent(
    role="Customer Support Ticket Classifier",
    goal=(
        "Accurately classify incoming support tickets by category "
        "and urgency level so they can be routed to the right team"
    ),
    backstory=(
        "You are a senior support operations specialist with 8 years "
        "of experience in customer support triage. You can quickly "
        "identify whether a ticket is a simple FAQ question or a "
        "complex issue requiring investigation."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

faq_agent = Agent(
    role="FAQ Support Specialist",
    goal=(
        "Quickly answer common customer questions using the FAQ "
        "database, providing clear and helpful responses"
    ),
    backstory=(
        "You are a friendly and efficient frontline support agent. "
        "You know the FAQ inside and out and can answer common "
        "questions in a warm, helpful tone."
    ),
    llm=llm,
    tools=[search_faq],
    verbose=True,
    allow_delegation=False,
)

research_agent = Agent(
    role="Support Research Specialist",
    goal=(
        "Thoroughly investigate complex customer issues by searching "
        "the knowledge base for relevant policies and procedures"
    ),
    backstory=(
        "You are a detail-oriented support investigator who leaves "
        "no stone unturned. When a customer has a billing dispute "
        "or technical problem, you dig into the knowledge base to "
        "find every relevant policy and procedure."
    ),
    llm=llm,
    tools=[search_knowledge_base],
    verbose=True,
    allow_delegation=False,
)

solution_agent = Agent(
    role="Customer Solution Specialist",
    goal=(
        "Write a clear, empathetic, and actionable response to the "
        "customer based on the research findings"
    ),
    backstory=(
        "You are a senior support agent known for turning frustrated "
        "customers into happy ones. You write responses that are "
        "empathetic, clear, and action-oriented."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)


# ============================================================
# 5. FLOW STATE (same as the notebook)
# ============================================================

class SupportTicketState(BaseModel):
    """State that flows through the entire support pipeline."""
    customer_name: str = ""
    customer_email: str = ""
    ticket_subject: str = ""
    ticket_message: str = ""
    category: str = ""
    complexity: str = ""
    urgency: str = ""
    resolution_subject: str = ""
    resolution_message: str = ""
    follow_up_needed: bool = False
    path_taken: str = ""


# ============================================================
# 6. THE FLOW (same as the notebook)
# ============================================================

class CustomerSupportFlow(Flow[SupportTicketState]):
    """
    The AI-powered customer support flow.
    Classifies → Routes → Resolves → Formats.
    """

    @start()
    def classify_ticket(self):
        """Entry point: classify the incoming ticket."""
        print(f"\nClassifying ticket from {self.state.customer_name}...")

        classify_task = Task(
            description=(
                f"Classify this customer support ticket:\n"
                f"  Customer: {self.state.customer_name}\n"
                f"  Subject: {self.state.ticket_subject}\n"
                f"  Message: {self.state.ticket_message}\n\n"
                f"Determine:\n"
                f"1. Category: password_reset, pricing, cancel_subscription, "
                f"integrations, billing_issue, technical_issue, or account_issue\n"
                f"2. Complexity: 'simple' if it's a common FAQ question, "
                f"'complex' if it requires investigation\n"
                f"3. Urgency: 'low', 'medium', or 'high'"
            ),
            expected_output="A classification with category, complexity, urgency, and reasoning.",
            agent=classifier_agent,
            output_pydantic=TicketClassification,
        )

        classify_crew = Crew(
            agents=[classifier_agent],
            tasks=[classify_task],
            process=Process.sequential,
            verbose=True,
        )
        classify_crew.kickoff()

        if classify_task.output.pydantic:
            classification = classify_task.output.pydantic
            self.state.category = classification.category
            self.state.complexity = classification.complexity
            self.state.urgency = classification.urgency

        print(f"Classification: {self.state.category} | {self.state.complexity} | urgency={self.state.urgency}")
        return self.state.complexity

    @router(classify_ticket)
    def route_ticket(self):
        """Route to 'simple' or 'complex' path."""
        if self.state.complexity == "simple":
            print(">> Routing to: SIMPLE path (FAQ)")
            return "simple"
        else:
            print(">> Routing to: COMPLEX path (investigation)")
            return "complex"

    @listen("simple")
    def handle_simple_ticket(self):
        """Handle simple tickets with FAQ agent."""
        print("Simple path: FAQ Agent handling ticket...")

        faq_task = Task(
            description=(
                f"Answer this customer support question using the FAQ database:\n"
                f"  Customer: {self.state.customer_name}\n"
                f"  Question: {self.state.ticket_message}\n\n"
                f"Search the FAQ for relevant information and write a friendly, "
                f"helpful response. Address the customer by name."
            ),
            expected_output="A helpful response with clear instructions.",
            agent=faq_agent,
            output_pydantic=SupportResponse,
        )

        faq_crew = Crew(
            agents=[faq_agent],
            tasks=[faq_task],
            process=Process.sequential,
            verbose=True,
        )
        faq_crew.kickoff()

        if faq_task.output.pydantic:
            response = faq_task.output.pydantic
            self.state.resolution_subject = response.subject
            self.state.resolution_message = response.message
            self.state.follow_up_needed = response.follow_up_needed

        self.state.path_taken = "simple"
        return self.state.resolution_message

    @listen("complex")
    def handle_complex_ticket(self):
        """Handle complex tickets with investigation crew."""
        print("Complex path: Investigation Crew handling ticket...")

        research_task = Task(
            description=(
                f"Investigate this customer support issue:\n"
                f"  Customer: {self.state.customer_name}\n"
                f"  Category: {self.state.category}\n"
                f"  Issue: {self.state.ticket_message}\n\n"
                f"Search the knowledge base for relevant policies, procedures, "
                f"and troubleshooting steps."
            ),
            expected_output="A detailed research report with relevant policies and recommended actions.",
            agent=research_agent,
        )

        solution_task = Task(
            description=(
                f"Based on the research findings, write a response to "
                f"{self.state.customer_name} about their issue:\n"
                f"  Issue: {self.state.ticket_message}\n\n"
                f"Guidelines:\n"
                f"- Acknowledge their frustration\n"
                f"- Explain what happened (if applicable)\n"
                f"- Provide clear next steps\n"
                f"- Be empathetic and professional"
            ),
            expected_output="A professional, empathetic customer response with clear next steps.",
            agent=solution_agent,
            context=[research_task],
            output_pydantic=SupportResponse,
        )

        investigation_crew = Crew(
            agents=[research_agent, solution_agent],
            tasks=[research_task, solution_task],
            process=Process.sequential,
            verbose=True,
        )
        investigation_crew.kickoff()

        if solution_task.output.pydantic:
            response = solution_task.output.pydantic
            self.state.resolution_subject = response.subject
            self.state.resolution_message = response.message
            self.state.follow_up_needed = response.follow_up_needed

        self.state.path_taken = "complex"
        return self.state.resolution_message

    @listen(or_(handle_simple_ticket, handle_complex_ticket))
    def format_final_response(self, resolution):
        """Runs after either path completes. Returns the final state."""
        print(f"Formatting final response (path: {self.state.path_taken})")
        return self.state


# ============================================================
# 7. THE MAIN FUNCTION (NEW — this is what the API calls)
# ============================================================

def process_ticket(
    customer_name: str,
    customer_email: str,
    ticket_subject: str,
    ticket_message: str,
) -> dict:
    """
    Process a support ticket through the CrewAI flow.

    This is the function that the API endpoint calls.
    It creates a flow, sets the ticket data, runs the flow,
    and returns the results as a dictionary.
    """
    # Create a new flow instance (fresh state for each ticket)
    flow = CustomerSupportFlow()

    # Set the ticket data on the flow state
    flow.state.customer_name = customer_name
    flow.state.customer_email = customer_email
    flow.state.ticket_subject = ticket_subject
    flow.state.ticket_message = ticket_message

    # Run the flow (this triggers the entire pipeline)
    flow.kickoff()

    # Return the results as a dictionary
    return {
        "customer_name": flow.state.customer_name,
        "customer_email": flow.state.customer_email,
        "category": flow.state.category,
        "complexity": flow.state.complexity,
        "urgency": flow.state.urgency,
        "path_taken": flow.state.path_taken,
        "resolution_subject": flow.state.resolution_subject,
        "resolution_message": flow.state.resolution_message,
        "follow_up_needed": flow.state.follow_up_needed,
    }

**The key difference from the notebook:** The `process_ticket()` function at the bottom. In the notebook, we manually created a flow and called `kickoff()`. Here, we wrap that into a function so the API endpoint can call it easily.

Everything else is identical to the notebook.

## Step 4: `routers/support.py` — The API Endpoint

This file defines the API endpoint that the frontend will call. It follows the **exact same pattern** as `routers/todos.py` in the todo app.

**Comparison:**

| Todo App (`routers/todos.py`) | Our App (`routers/support.py`) |
|------------------------------|-------------------------------|
| `@router.post("")` creates a todo | `@router.post("/ticket")` processes a ticket |
| Calls `crud.create_todo(db, todo)` | Calls `crew_logic.process_ticket(...)` |
| Returns the saved todo | Returns the AI response |

Same pattern: receive a request → process it → return the result.

In [ ]:
%%writefile 074-crewai-backend/routers/support.py
from fastapi import APIRouter
import schemas
import crew_logic

router = APIRouter(
    prefix="/support"
)


@router.post("/ticket", response_model=schemas.TicketResponse)
async def process_support_ticket(ticket: schemas.TicketRequest):
    """
    Process a customer support ticket through the CrewAI flow.

    The frontend sends a ticket with:
      - customer_name, customer_email, ticket_subject, ticket_message

    The backend runs the CrewAI flow and returns:
      - category, complexity, urgency, resolution_message, etc.
    """
    result = crew_logic.process_ticket(
        customer_name=ticket.customer_name,
        customer_email=ticket.customer_email,
        ticket_subject=ticket.ticket_subject,
        ticket_message=ticket.ticket_message,
    )
    return result

**That's it!** Just 20 lines of code. The endpoint:
1. Receives a `TicketRequest` from the frontend.
2. Calls `crew_logic.process_ticket()` which runs the entire CrewAI Flow.
3. Returns the result as a `TicketResponse`.

All the heavy lifting happens inside `crew_logic.py`.

## Step 5: `main.py` — The FastAPI App

This is the entry point of our backend. It creates the FastAPI app, adds CORS (so the frontend can talk to the backend), and includes our router.

This is almost identical to the todo app's `main.py`.

In [ ]:
%%writefile 074-crewai-backend/main.py
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import PlainTextResponse
from starlette.exceptions import HTTPException as StarletteHTTPException

app = FastAPI()

# Include the support router
from routers import support
app.include_router(support.router)

# CORS configuration — allows the frontend (running on port 3000) to call our API
app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:3000"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


# Global error handler
@app.exception_handler(StarletteHTTPException)
async def http_exception_handler(request, exc):
    return PlainTextResponse(str(exc.detail), status_code=exc.status_code)


# Root endpoint — just to check if the server is running
@app.get("/")
def read_root():
    return {"message": "CrewAI Support Backend is running!"}

**Comparison with the todo app's `main.py`:**

| Todo App | Our App |
|----------|--------|
| `from routers import todos` | `from routers import support` |
| `app.include_router(todos.router)` | `app.include_router(support.router)` |
| Uses `config.py` for settings | Simpler — no database settings needed |
| CORS allows `localhost:3000` | Same — CORS allows `localhost:3000` |

Almost identical. The only difference is we don't need database configuration.

## Step 6: Create the `.env` File

In [ ]:
%%writefile 074-crewai-backend/.env
OPENAI_API_KEY=your-openai-api-key-here

**Important:** Replace `your-openai-api-key-here` with your actual OpenAI API key before running the server.

## Step 7: How to Run the Backend

To start the backend server, open a terminal, navigate to the `074-crewai-backend/` folder, and run:

```bash
uvicorn main:app --reload
```

This starts the server at `http://localhost:8000`.

- `main` refers to `main.py`.
- `app` refers to the `app = FastAPI()` object inside `main.py`.
- `--reload` makes the server restart automatically when you change the code (useful during development).

You should see something like:

```
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Started reloader process
```

## Step 8: Test the API

Once the server is running, you can test it in three ways:

### Option A: Use the automatic docs (easiest)

FastAPI automatically generates interactive documentation. Open your browser and go to:

```
http://localhost:8000/docs
```

You'll see the `/support/ticket` endpoint. Click on it, then click "Try it out", paste the JSON below, and click "Execute".

### Option B: Use `curl` from the terminal

```bash
curl -X POST http://localhost:8000/support/ticket \
  -H "Content-Type: application/json" \
  -d '{
    "customer_name": "Alice Johnson",
    "customer_email": "alice@example.com",
    "ticket_subject": "Can'\''t log in",
    "ticket_message": "Hi, I forgot my password. How do I reset it?"
  }'
```

### Option C: Use Python (from this notebook or any script)

Make sure the server is running in a terminal first, then run the cell below.

In [ ]:
import requests

# Make sure the server is running first! (uvicorn main:app --reload)
response = requests.post(
    "http://localhost:8000/support/ticket",
    json={
        "customer_name": "Alice Johnson",
        "customer_email": "alice@example.com",
        "ticket_subject": "Can't log in — need password reset",
        "ticket_message": "Hi, I forgot my password and I can't log into my account. How do I reset it? Thanks!",
    },
)

if response.ok:
    data = response.json()
    print(f"Category: {data['category']}")
    print(f"Complexity: {data['complexity']}")
    print(f"Urgency: {data['urgency']}")
    print(f"Path taken: {data['path_taken']}")
    print(f"Follow-up needed: {data['follow_up_needed']}")
    print(f"\n--- Response to customer ---")
    print(f"Subject: {data['resolution_subject']}")
    print(f"\n{data['resolution_message']}")
else:
    print(f"Error: {response.status_code}")
    print(response.text)

## Recap: From Notebook to Backend

Here's a summary of what we did and how each file maps to the original notebook and the todo app:

### How the Files Connect

```
Frontend (browser)               Backend (server)
┌──────────────────┐             ┌─────────────────────────────────────┐
│                  │  POST       │  main.py                            │
│  Submit Ticket   │ ─────────>  │    └── routers/support.py           │
│  Form            │  JSON       │          └── crew_logic.py          │
│                  │             │               ├── Tools             │
│                  │             │               ├── Agents            │
│                  │  JSON       │               ├── Flow              │
│  Display         │ <─────────  │               └── process_ticket()  │
│  Response        │             │                                     │
└──────────────────┘             └─────────────────────────────────────┘
```

### File-by-File Summary

| File | What It Does | Inspired By |
|------|-------------|------------|
| `main.py` | Creates FastAPI app, adds CORS, includes router | `050-todoapp-LC-backend/main.py` |
| `schemas.py` | Defines request/response data shapes | `050-todoapp-LC-backend/schemas.py` |
| `routers/support.py` | API endpoint that receives tickets | `050-todoapp-LC-backend/routers/todos.py` |
| `crew_logic.py` | All CrewAI code (agents, tools, flow) | `037-5th-top-use-case-crewai-2026.ipynb` |
| `.env` | API key storage | `050-todoapp-LC-backend/.env` |

### What We Didn't Need (and Why)

The todo app had several files we **don't need**:

| Todo App File | Why We Don't Need It |
|--------------|---------------------|
| `database.py` | No database — we process tickets on the fly |
| `models.py` | No database models — no data to store |
| `crud.py` | No CRUD operations — no create/read/update/delete |
| `config.py` | No database settings — just need the OpenAI API key |
| `alembic/` | No database migrations — no database at all |

Our backend is simpler because it doesn't store anything. It receives a ticket, processes it with CrewAI, and returns the response.

## What's Next?

Now that we have a working backend, the next step is to build a **frontend** — a web page where users can:
1. Fill in a form with their name, email, subject, and message.
2. Click "Submit Ticket".
3. See the AI's response displayed on the page.

That frontend will call our `POST /support/ticket` endpoint, just like the todo app's frontend calls `POST /todos`.